Notebook: feature_engineering.ipynb  
Project: EcoPackAI  

This notebook creates engineered sustainability and performance features:
- CO₂ Impact Index (CII)
- Cost Efficiency Index (CEI)
- Material Suitability Score (MSS)


In [ ]:
# cell 1 Environment setup
import pandas as pd
import numpy as np


In [ ]:
#cell 2 Data loading
material = pd.read_csv(r'C:\Users\Ranjit\OneDrive\Desktop\AI-Powered-Sustainable-Packaging-Recommendation-System\ml\data\final\processed\material_cleaned.csv')
print("material and product data loaded successfully")
print(material.shape)
material.head()


In [ ]:
#cell 3 checking missing values
material.isnull().sum()

In [ ]:
# cell 4- Safe Min-Max Normalization
def min_max_safe(series):
    if series.max() == series.min():
        return np.zeros(len(series))
    return (series - series.min()) / (series.max() - series.min())


In [ ]:
# cell 5 — CO₂ Impact Index 
co2_norm = 1 - min_max_safe(material["co2 emission per kg (estimated)"])
bio_norm = min_max_safe(material["biodegradation time (days)"])
recycle_norm = min_max_safe(material["recyclability (%)"])

material["co2_impact_index"] = (
    0.4 * co2_norm +
    0.3 * bio_norm +
    0.3 * recycle_norm
) * 100

material["co2_impact_index"] = material["co2_impact_index"].clip(0, 100)




In [ ]:
# cell 6 Cost Efficiency Index
cost_norm = 1 - min_max_safe(material["cost per unit (usd)"])
weight_norm = min_max_safe(material["total material weight (tons)"])
strength_norm = min_max_safe(material["strength_mpa"])

material["cost_efficiency_index"] = (
    0.4 * cost_norm +
    0.3 * weight_norm +
    0.3 * strength_norm
) * 100

material["cost_efficiency_index"] = material["cost_efficiency_index"].clip(0, 100)




In [ ]:
#cell 7 Material Suitability Score
strength_norm_mss = min_max_safe(material["strength_mpa"])
weight_norm_mss = min_max_safe(material["weight_capacity"])

# Base numeric suitability score
base_mss = (
    0.6 * strength_norm_mss +
    0.4 * weight_norm_mss
)

# Industry boost (normalized influence)
industry_boost = {
    "Pharmacy": 1.0,
    "Electronics": 0.8,
    "Household": 0.5,
    "Apparel": 0.4
}

industry_factor = material["industry_use_case"].map(industry_boost).fillna(0.3)

# Final MSS
material["material_suitability_score"] = (
    0.85 * base_mss +
    0.15 * industry_factor
) * 100

material["material_suitability_score"] = material["material_suitability_score"].clip(0, 100)



In [ ]:
#cell 8
material[
    ["co2_impact_index",
     "cost_efficiency_index",
     "material_suitability_score"]
].describe()



In [ ]:
#cell 9 Save engineered features
material.to_csv("../data/processed/material_engineered.csv", index=False)
